In [1]:
!pip install requests beautifulsoup4 pandas

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Phase 2: Scraping
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
all_books_data = []

print("Starting scraping process...")

for page_num in range(1, 6):
    print(f"Scraping page {page_num}...")
    url = base_url.format(page_num)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = soup.find_all('article', class_='product_pod')

    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text
        rating = book.find('p', class_='star-rating')['class'][1]
        availability = book.find('p', class_='instock availability').text.strip()

        all_books_data.append({
            'Title': title,
            'Price': price,
            'Rating': rating,
            'Availability': availability
        })

    time.sleep(1)

# Phase 3: Data Cleaning
print("\nScraping complete! Cleaning data...")
df = pd.DataFrame(all_books_data)

# 1. FIXED: Clean Price using Regex (removes 'Â', '£', and any other weird characters)
df['Price'] = df['Price'].str.replace(r'[^\d.]', '', regex=True).astype(float)

# 2. Map Ratings to Numbers
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['Rating'] = df['Rating'].map(rating_map)

# 3. Check for missing data
print(f"\nMissing values:\n{df.isnull().sum()}")

# Phase 4: Export
df.to_csv('clean_books_data.csv', index=False)

print(f"\nSuccess! Scraped {len(df)} books.")
print("Here is a preview of your clean dataset:")
print(df.head())

Starting scraping process...
Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...

Scraping complete! Cleaning data...

Missing values:
Title           0
Price           0
Rating          0
Availability    0
dtype: int64

Success! Scraped 100 books.
Here is a preview of your clean dataset:
                                   Title  Price  Rating Availability
0                   A Light in the Attic  51.77       3     In stock
1                     Tipping the Velvet  53.74       1     In stock
2                             Soumission  50.10       1     In stock
3                          Sharp Objects  47.82       4     In stock
4  Sapiens: A Brief History of Humankind  54.23       5     In stock
